# Clase 11 — Taller de simulación y Monte Carlo
## Cuaderno del estudiante

**Casos:** promesa de entrega, comparación de inventarios y diseño por precisión.  
**Regla del taller:** primero se formula y se anticipa; después se consulta al agente de IA; finalmente se verifica, interpreta y defiende el resultado.

> Este cuaderno se puede ejecutar de principio a fin aun antes de completar los `TODO`. Las celdas incompletas informarán `PENDIENTE` en lugar de interrumpir la ejecución.

## Protocolo obligatorio de trabajo con IA

### Antes de abrir el agente
1. Escriba la pregunta de decisión en una oración.
2. Defina variables, unidades, horizonte, distribuciones y dependencias.
3. Declare el **estimando**: la cantidad del modelo que quiere conocer.
4. Escriba el estimador y anticipe rangos, signos y valores de referencia.
5. Proponga al menos dos pruebas que podrían revelar un error.

### Durante el uso del agente
1. Entregue el contrato anterior y pida código visible y explicable.
2. Exija un generador aleatorio explícito, semilla reproducible y controles automáticos.
3. No acepte una biblioteca, fórmula o línea que el equipo no pueda explicar.
4. Ejecute casos pequeños a mano y compárelos con el código.

### Después de usar el agente
1. Ejecute las pruebas y contraste con referencias analíticas cuando existan.
2. Distinga error Monte Carlo de incertidumbre del modelo.
3. Redacte la conclusión con unidades, intervalo y supuestos.
4. Registre qué propuso la IA, qué corrigió el equipo y qué limitación permanece.

## Productos que debe entregar el equipo

Para cada ciclo conserve en el cuaderno:

- ficha del modelo completada **antes** del agente;
- prompt enviado y fragmentos relevantes de la respuesta;
- código final comentado;
- controles ejecutados y cualquier corrección realizada;
- respuesta a las preguntas interpretativas;
- una idea para la puesta en común.

Los roles son: **modelador**, **operador del agente** y **auditor**. Deben rotar en cada ciclo.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import t as student_t
    SCIPY_DISPONIBLE = True
except Exception:
    student_t = None
    SCIPY_DISPONIBLE = False

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('Entorno listo. SciPy disponible:', SCIPY_DISPONIBLE)

In [ ]:
def valor_critico_95(gl):
    """Valor crítico bilateral aproximado para un IC del 95 %."""
    if SCIPY_DISPONIBLE:
        return float(student_t.ppf(0.975, gl))
    tabla = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571,
             6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228,
             20: 2.086, 30: 2.042}
    if gl in tabla:
        return tabla[gl]
    return 1.96 if gl >= 30 else 2.262

def informar_pendiente(nombre):
    print(f'PENDIENTE: complete {nombre} y vuelva a ejecutar esta celda.')

def comprobar(condicion, mensaje):
    estado = 'OK' if bool(condicion) else 'REVISAR'
    print(f'[{estado}] {mensaje}')
    return bool(condicion)

# Ciclo 1 — ¿Conviene prometer una entrega en 40 minutos?

Una plataforma modela el tiempo de preparación del restaurante como $R\sim U(10,20)$ minutos y el viaje como $V\sim U(20,40)$ minutos. En el modelo base se supone independencia y $T=R+V$.

## Ficha previa al agente — complete en texto

- **Decisión:** ...
- **Población y horizonte:** ...
- **Variables y unidades:** ...
- **Origen que deberían tener los rangos y la forma uniforme:** ...
- **Supuesto de dependencia:** ...
- **Estimandos:** $E[T]$, $SD(T)$, $P(T>40)$ y $q_{0.90}(T)$.
- **Estimadores correspondientes:** ...
- **Rangos que toda réplica debe respetar:** ...
- **Dos resultados que refutarían el programa:** ...

Antes de programar, calcule o justifique como referencias: $E[T]=45$, $SD(T)=\sqrt{500/12}$ y $P(T>40)=0.75$.

## Contrato sugerido para el agente — Ciclo 1

Copie su ficha completa y agregue:  

> Implemente una función `simular_entregas(n, semilla)` con `numpy.random.default_rng`. Debe devolver un `DataFrame` con columnas `R`, `V`, `T` e `incumple`. No reinicie la semilla dentro de cada réplica. Agregue controles de rangos, de la identidad $T=R+V$ y de reproducibilidad. Explique cada cálculo y no oculte advertencias.

Pegue aquí, como comentario o celda nueva, el prompt finalmente utilizado y anote qué cambió respecto de la primera respuesta del agente.

In [ ]:
SEMILLA_ENTREGAS = 20261019
N_ENTREGAS = 50_000

def simular_entregas(n, semilla):
    # TODO 1.1: cree un único generador; genere R y V; calcule T e incumple.
    # Retorno esperado: DataFrame con n filas y columnas R, V, T, incumple.
    return None

entregas = simular_entregas(N_ENTREGAS, SEMILLA_ENTREGAS)
if entregas is None:
    informar_pendiente('simular_entregas')
else:
    display(entregas.head())

In [ ]:
def resumen_entregas(df):
    # TODO 1.2: calcule media y SD muestral de T, proporción T>40 y percentil 0.90.
    # Agregue IC del 95 % para la media y para la proporción.
    return None

resumen_1 = None if entregas is None else resumen_entregas(entregas)
if resumen_1 is None:
    informar_pendiente('resumen_entregas')
else:
    display(pd.Series(resumen_1, name='estimación'))

In [ ]:
# CONTROLES EXPLÍCITOS DEL CICLO 1
if entregas is None:
    informar_pendiente('simular_entregas para ejecutar los controles')
else:
    comprobar(len(entregas) == N_ENTREGAS, 'se generó exactamente N observaciones')
    comprobar(entregas['R'].between(10, 20).all(), '10 <= R <= 20')
    comprobar(entregas['V'].between(20, 40).all(), '20 <= V <= 40')
    comprobar(entregas['T'].between(30, 60).all(), '30 <= T <= 60')
    comprobar(np.allclose(entregas['T'], entregas['R'] + entregas['V']), 'T = R + V')
    repetida = simular_entregas(N_ENTREGAS, SEMILLA_ENTREGAS)
    comprobar(repetida is not None and entregas.equals(repetida), 'misma semilla, mismo resultado')
    comprobar(abs(entregas['T'].mean() - 45) < 0.15, 'media próxima al valor analítico 45')
    comprobar(abs(entregas['T'].std(ddof=1) - math.sqrt(500/12)) < 0.15,
              'SD próxima a sqrt(500/12)')
    comprobar(abs(entregas['incumple'].mean() - 0.75) < 0.01,
              'probabilidad próxima a 0.75')

## Interpretación y puesta en común — Ciclo 1

Responda con oraciones completas:

1. ¿Qué significa operacionalmente la estimación de $P(T>40)$?
2. ¿El intervalo cuantifica incertidumbre del sistema real o error Monte Carlo bajo este modelo?
3. ¿Qué evidencia real necesitaría para defender las dos distribuciones uniformes?
4. ¿Qué podría cambiar si $R$ y $V$ estuvieran asociados positivamente?
5. Escriba una recomendación gerencial de dos oraciones que incluya una limitación.

**Puesta en común (7 min):** muestre un control que aumentó su confianza y un supuesto que todavía limita la decisión.

# Ciclo 2 — Comparación pareada de políticas de inventario

Una tienda decide $q$ antes de observar la demanda semanal $D$. El excedente cuesta $h=1$ por unidad y el faltante $p=5$ por unidad:

$$C(q,D)=h(q-D)^+ + p(D-q)^+.$$

Se comparan $q_A=120$ y $q_B=140$ sobre las mismas diez semanas. Definimos $\Delta=C_B-C_A$: un valor negativo favorece a $q_B$.

## Ficha previa al agente

- Explique qué representa una unidad de $q$, $D$, $h$, $p$ y $C$.
- Calcule a mano $C(120,100)$ y $C(120,140)$.
- Anticipe el signo de $\Delta$ cuando $D=80$ y cuando $D=140$.
- Explique por qué ambas políticas deben evaluarse con la misma demanda.
- Declare el estimando de la comparación y la regla de decisión.

In [ ]:
DEMANDA_PILOTO = np.array([80, 100, 120, 140, 120, 140, 100, 120, 140, 80], dtype=float)
Q_A, Q_B = 120.0, 140.0
H, P = 1.0, 5.0
print('Demanda piloto:', DEMANDA_PILOTO.tolist())

## Contrato sugerido para el agente — Ciclo 2

> Implemente `costo_inventario(q, demanda, h, p)` de manera vectorizada y `comparar_politicas(demanda, q_a, q_b, h, p)`. La segunda función debe devolver costos y diferencias pareadas, junto con media, SD, SE e IC t bilateral del 95 % para $E[C_B-C_A]$. Incluya pruebas de borde derivadas de la fórmula y explique por qué no corresponde comparar dos intervalos independientes.

In [ ]:
def costo_inventario(q, demanda, h=1.0, p=5.0):
    # TODO 2.1: implemente h*max(q-D,0) + p*max(D-q,0).
    # Debe admitir un escalar o un arreglo como demanda.
    return None

def comparar_politicas(demanda, q_a, q_b, h=1.0, p=5.0):
    # TODO 2.2: calcule C_A, C_B, delta=C_B-C_A, resumen e IC pareado.
    return None

comparacion = comparar_politicas(DEMANDA_PILOTO, Q_A, Q_B, H, P)
if comparacion is None:
    informar_pendiente('costo_inventario y comparar_politicas')
else:
    display(comparacion['tabla'])
    display(pd.Series(comparacion['resumen'], name='valor'))

In [ ]:
# CONTROLES Y PRUEBAS DE BORDE DEL CICLO 2
if costo_inventario(10, 10, H, P) is None:
    informar_pendiente('costo_inventario para ejecutar las pruebas')
else:
    comprobar(np.isclose(costo_inventario(10, 10, H, P), 0), 'si q=D, el costo es cero')
    comprobar(np.isclose(costo_inventario(12, 10, 0, P), 0),
              'si h=0 y q>=D, el excedente no cuesta')
    comprobar(np.isclose(costo_inventario(8, 10, H, 0), 0),
              'si p=0 y q<=D, el faltante no cuesta')
    vector_prueba = np.array([80, 100, 120, 140], dtype=float)
    costos_prueba = np.asarray(costo_inventario(120, vector_prueba, H, P))
    comprobar((costos_prueba >= 0).all(), 'los costos son no negativos')
    comprobar(np.allclose(costos_prueba, [40, 20, 0, 100]),
              'los cuatro casos manuales coinciden')

In [ ]:
def evaluar_grilla(q_valores, demanda, h=1.0, p=5.0):
    # TODO 2.3: use LA MISMA demanda para todos los q y devuelva un DataFrame
    # con columnas q y costo_medio. No genere una muestra distinta para cada q.
    return None

Q_GRILLA = np.arange(80, 161, 5)
grilla = evaluar_grilla(Q_GRILLA, DEMANDA_PILOTO, H, P)
if grilla is None:
    informar_pendiente('evaluar_grilla')
else:
    display(grilla)
    mejor_fila = grilla.loc[grilla['costo_medio'].idxmin()]
    print('Candidato muestral:', mejor_fila.to_dict())
    ax = grilla.plot(x='q', y='costo_medio', marker='o', legend=False)
    ax.set(xlabel='Cantidad q', ylabel='Costo medio estimado',
           title='Curva de decisión sobre la muestra piloto')
    ax.grid(alpha=0.3)
    plt.show()

## Interpretación y puesta en común — Ciclo 2

1. ¿Qué significa que la media de $\Delta$ sea negativa?
2. ¿Por qué una media negativa no alcanza si el intervalo contiene cero?
3. ¿Qué información aporta $s_\Delta$ y cuál aporta $SE(\bar\Delta)$?
4. ¿Por qué el candidato que minimiza el costo sobre la grilla debería evaluarse con datos nuevos?
5. ¿Qué problema aparece si las ventas históricas están censuradas por quiebres de stock?

**Puesta en común (7 min):** explique la diferencia entre *preferencia muestral* y *evidencia concluyente*.

# Ciclo 3 — Elegir el número de réplicas por precisión

Queremos estimar $E[T]$ para el caso de entregas con una semiamplitud objetivo $h^*$ en un IC bilateral del 95 %. Un piloto de $N_0$ réplicas estima $s_T$ y permite aproximar

$$N\approx\left\lceil\left(\frac{t_{0.975,N_0-1}s_T}{h^*}\right)^2\right\rceil.$$

El valor $t_{0.975,N_0-1}$ es el cuantil de una distribución t de Student que deja 2,5 % en la cola superior. Depende de los grados de libertad $N_0-1$ y se aproxima a $1.96$ cuando la muestra es grande.

## Ficha previa al agente

- **Estimando y unidad:** ...
- **Nivel de confianza:** ...
- **Semiamplitud máxima tolerada:** ...
- **Por qué el piloto y la muestra final deben ser independientes:** ...
- **Qué hará si la precisión final no se alcanza:** ...

## Contrato sugerido para el agente — Ciclo 3

> Diseñe un experimento piloto y uno final independientes para estimar la media de $T$. Use una semilla maestra y `SeedSequence.spawn` o semillas documentadas. Calcule el tamaño final con la fórmula dada, informe el IC alcanzado y compruebe automáticamente si la semiamplitud es menor o igual que $h^*$. No implemente una parada opcional que mire repetidamente el intervalo hasta que resulte conveniente.

In [ ]:
SEMILLA_PRECISION = 20261022
N0 = 400
H_OBJETIVO = 0.10  # minutos

def planificar_n_desde_piloto(s_piloto, n0, h_objetivo):
    # TODO 3.1: use valor_critico_95(n0-1), eleve al cuadrado y redondee hacia arriba.
    return None

def ejecutar_diseno_precision(n0, h_objetivo, semilla):
    # TODO 3.2:
    # 1) obtenga dos subflujos independientes desde una semilla maestra;
    # 2) ejecute el piloto y estime s;
    # 3) calcule N;
    # 4) ejecute una muestra final independiente;
    # 5) devuelva media, s, SE, IC, semiamplitud y alcanzado.
    return None

diseno = ejecutar_diseno_precision(N0, H_OBJETIVO, SEMILLA_PRECISION)
if diseno is None:
    informar_pendiente('planificar_n_desde_piloto y ejecutar_diseno_precision')
else:
    display(pd.Series(diseno, name='resultado'))

In [ ]:
# CONTROLES EXPLÍCITOS DEL CICLO 3
if diseno is None:
    informar_pendiente('el diseño por precisión para ejecutar los controles')
else:
    comprobar(diseno['N_final'] >= 2, 'el tamaño final es válido')
    comprobar(diseno['IC_inferior'] <= diseno['media_final'] <= diseno['IC_superior'],
              'el intervalo está centrado en la media estimada')
    comprobar(np.isclose((diseno['IC_superior'] - diseno['IC_inferior']) / 2,
                         diseno['semiamplitud']),
              'la semiamplitud coincide con el intervalo')
    comprobar(diseno['semiamplitud'] <= H_OBJETIVO,
              'se alcanzó la precisión objetivo')

## Interpretación y puesta en común — Ciclo 3

1. ¿Qué dato del piloto determina principalmente el tamaño final?
2. Si se desea reducir $h^*$ a la mitad, ¿aproximadamente por qué factor cambia $N$?
3. ¿Una semiamplitud pequeña demuestra que el modelo representa la realidad? Justifique.
4. ¿Qué diferencia hay entre fijar $N$ con un piloto y detenerse después de mirar repetidamente el resultado?
5. ¿El costo computacional obtenido es razonable para esta decisión?

**Puesta en común (7 min):** compare tamaño muestral, precisión y costo. Indique además qué fuente de incertidumbre no se redujo al aumentar $N$.

# Auditoría final y ticket de salida

## Lista de control antes de entregar

- [ ] Las tres fichas fueron completadas antes de consultar al agente.
- [ ] Todas las variables tienen significado y unidades.
- [ ] Las semillas y tamaños muestrales quedaron registrados.
- [ ] Los controles y las pruebas de borde se ejecutaron.
- [ ] Cada intervalo responde exactamente a la pregunta formulada.
- [ ] Se distingue verificación del código de validación del modelo.
- [ ] Las conclusiones declaran sus supuestos y límites.
- [ ] Se documentaron las correcciones realizadas a las propuestas de la IA.

## Ticket de salida

Una comparación produce $\bar\Delta=-0.2$ con error estándar pareado $0.4$.

1. ¿Puede declararse una ganadora con un criterio bilateral del 95 %?
2. ¿Cuántas veces más réplicas se necesitan para reducir el error estándar a $0.2$?
3. Escriba una advertencia gerencial que distinga precisión Monte Carlo de validez del modelo.